# StayNest - Session 6 Assignment (PySpark Deep Dive)
Work through the 8 tasks below in order. Read the Assignment Questions PDF for the
full detail and acceptance criteria. Fill in each `# TODO` cell, run it, and keep the
output visible. Run on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `customers.csv` to a Volume, then set `BASE`
to that path and run this cell. Counts should be 12000 / 200 / 2000.

In [0]:
# Point BASE at YOUR Volume path
BASE = "/Volumes/workspace/default/staynest"

print(spark.version)

read_csv = lambda name: (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df   = read_csv("bookings")
hotels_df     = read_csv("hotels")
customers_df  = read_csv("customers")

print(f"bookings: {bookings_df.count()}, "
      f"hotels: {hotels_df.count()}, "
      f"customers: {customers_df.count()}")

4.2.0
bookings: 12000, hotels: 200, customers: 2000


## Task 1 - Read and inspect
Show the schema, a few sample rows, the row count, and summary stats for the
numeric columns of `bookings_df`.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    DoubleType,
    IntegerType
)

bookings_schema = StructType([
    StructField("booking_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=False),
    StructField("hotel_id", IntegerType(), nullable=False),
    StructField("booking_date", DateType(), nullable=False),
    StructField("city", StringType(), nullable=False),
    StructField("nights", IntegerType(), nullable=False),
    StructField("amount", DoubleType(), nullable=False),
    StructField("status", StringType(), nullable=False)
])

booking_df_prod = (
    spark.read
    .schema(bookings_schema)
    .option("header", True)
    .csv(f"{BASE}/bookings.csv")
)

booking_df_prod.printSchema()
booking_df_prod.count()
booking_df_prod.describe().show(5)

root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+-------+------------------+-----------------+-----------------+---------+------------------+------------------+---------+
|summary|        booking_id|      customer_id|         hotel_id|     city|            nights|            amount|   status|
+-------+------------------+-----------------+-----------------+---------+------------------+------------------+---------+
|  count|             12000|            12000|            12000|    12000|             12000|             12000|    12000|
|   mean|         9005999.5|     700999.56125|       3097.86275|     NULL| 3.994083333333333|19367.160963333372|     NULL|
| stddev|3464.2459496981446|579.5736502376229|58.035617714

## Task 2 - Select and filter
From `bookings_df`, select a few useful columns and return the **completed**
bookings with `amount` over 10000 in the cities Goa or Mumbai. Use `col()`, combine
conditions with `&`, and use `.isin(...)`.

In [0]:
# TODO
from pyspark.sql.functions import col

bookings_selected = (
    bookings_df
    .select(
        col("booking_id"),
        col("city"),
        col("amount")
    )
    .filter(
        (col("city").isin("Goa", "Mumbai")) &
        (col("status") == "completed") &
        (col("amount") > 10000)
    )
)

bookings_selected.show(10)

+----------+------+--------+
|booking_id|  city|  amount|
+----------+------+--------+
|   9000007|Mumbai|15693.64|
|   9000010|   Goa|30786.36|
|   9000028|   Goa|27008.87|
|   9000034|   Goa|64388.77|
|   9000050|Mumbai|24275.14|
|   9000061|Mumbai|19180.93|
|   9000070|   Goa|58790.91|
|   9000071|Mumbai|47548.35|
|   9000075|   Goa|18043.52|
|   9000087|   Goa|21529.88|
+----------+------+--------+
only showing top 10 rows


## Task 3 - Derived columns
Add: `amount_with_gst` (amount plus 12% tax), a `value_tier`
(premium / standard / budget) using `when`/`otherwise`, and a `booking_month`
from `booking_date`.

In [0]:
# TODO
from pyspark.sql.functions import when,month
bookings_df=bookings_df.withColumn(
    "amount_with_gst",
    col("amount")*1.12
    )

bookings_df=bookings_df.withColumn(
    "value_tier",
    when(col("amount")>12000,"Premium")
    .when(col("amount")>8000,"Standard")
    .otherwise("Budget")
    )

bookings_df=bookings_df.withColumn(
    "bookings_month",
    month(col("booking_date")
    )

    )
display(bookings_df)


booking_id,customer_id,hotel_id,booking_date,city,nights,amount,status,amount_with_gst,value_tier,bookings_month
9000000,701600,3095,2025-11-27,Jaipur,4,6087.65,completed,6818.168000000001,Budget,11
9000001,700065,3057,2025-11-06,Delhi,1,8211.19,cancelled,9196.5328,Standard,11
9000002,701392,3187,2025-08-21,Jaipur,2,7176.52,cancelled,8037.702400000001,Budget,8
9000003,700867,3112,2025-03-22,Bengaluru,5,7880.62,completed,8826.2944,Budget,3
9000004,701521,3043,2025-04-19,Mumbai,5,21021.51,pending,23544.0912,Premium,4
9000005,701998,3018,2025-10-14,Mumbai,2,18124.49,cancelled,20299.428800000005,Premium,10
9000006,701336,3012,2025-11-25,Delhi,7,70999.15,completed,79519.048,Premium,11
9000007,700868,3127,2025-04-10,Mumbai,2,15693.64,completed,17576.876800000002,Premium,4
9000008,700687,3045,2025-01-15,Mumbai,1,1492.62,completed,1671.7344,Budget,1
9000009,700022,3040,2025-06-08,Goa,5,6694.39,completed,7497.716800000001,Budget,6


## Task 4 - Aggregations
For **completed** bookings, group by `city` and return: number of bookings, total
revenue, average amount, biggest booking, and the count of unique customers.
Order by revenue, highest first.

In [0]:
from pyspark.sql.functions import col, count, sum, avg, max, countDistinct

city_revenue = (
    bookings_df
    .filter(col("status") == "completed")
    .groupBy("city")
    .agg(
        count("booking_id").alias("no_of_bookings"),
        sum("amount").alias("total_revenue"),
        avg("amount").alias("avg_revenue"),
        max("amount").alias("biggest_order"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(col("total_revenue").desc())
)

display(city_revenue)

city,no_of_bookings,total_revenue,avg_revenue,biggest_order,unique_customers
Goa,2546,4.459670178999999E7,17516.379336213664,78481.24,1441
Mumbai,1715,3.624122112E7,21131.90735860058,78524.52,1153
Delhi,1174,2.631428154000001E7,22414.209148211252,78669.69,861
Jaipur,979,2.4436853129999984E7,24961.03486210417,76904.87,796
Bengaluru,1318,2.267013697000002E7,17200.4074127466,78568.81,969
Udaipur,691,1.2094427419999994E7,17502.78931982633,77939.5,592
Rishikesh,407,8606121.57999999,21145.261867321846,77458.26,363
Manali,480,6235480.680000003,12990.584750000007,77291.17,409
Munnar,244,3979216.1100000013,16308.262745901644,65177.15,233
Anantapur,106,2257080.65,21293.213679245284,78237.19,105


## Task 5 - Joins
Inner-join bookings to hotels to enrich each booking. Do a left join too. Use
`left_anti` to check for orphaned bookings (expect 0). Then do a three-way join
with customers.

In [0]:
# TODO

# Inner join: bookings + hotels
bookings_hotels_df = bookings_df.join(
    hotels_df,
    on="hotel_id",
    how="inner"
)

print("Inner join count:", bookings_hotels_df.count())
bookings_hotels_df.show()

# Left join: bookings + hotels
bookings_hotels_left_df = bookings_df.join(
    hotels_df,
    on="hotel_id",
    how="left"
)

print("Left join count:", bookings_hotels_left_df.count())

# Left anti join: find orphaned bookings
orphans_df = bookings_df.join(
    hotels_df,
    on="hotel_id",
    how="left_anti"
)

print("Orphaned bookings:", orphans_df.count())

# Three-way join: bookings + hotels + customers
bookings_hotels_customers_df = (
    bookings_df
    .join(
        hotels_df,
        on="hotel_id",
        how="inner"
    )
    .join(
        customers_df,
        on="customer_id",
        how="inner"
    )
)

print("Three-way join count:", bookings_hotels_customers_df.count())
bookings_hotels_customers_df.show()



Inner join count: 12000
+--------+----------+-----------+------------+---------+------+--------+---------+------------------+----------+--------------+----------------+---------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|     city|nights|  amount|   status|   amount_with_gst|value_tier|bookings_month|      hotel_name|     city|category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+------------------+----------+--------------+----------------+---------+--------+-----------+
|    3095|   9000000|     701600|  2025-11-27|   Jaipur|     4| 6087.65|completed| 6818.168000000001|    Budget|            11|   Orchid Suites|   Jaipur|  Budget|        3.8|
|    3057|   9000001|     700065|  2025-11-06|    Delhi|     1| 8211.19|cancelled|         9196.5328|  Standard|            11|  Orchid Retreat|    Delhi|  Luxury|        4.1|
|    3187|   9000002|     701392|  2025-08-21|   Jaipur|     2| 7176.52|cancelled| 8037.70240000

## Task 6 - Spark SQL + a window function
Register temp views and use `spark.sql` to get revenue by hotel `category` for
completed bookings. Then use a window function to rank the **top 3 hotels by
revenue within each city**.

In [0]:
bookings_df.createOrReplaceTempView("bookings")

hotels_df.createOrReplaceTempView("hotels")

customers_df.createOrReplaceTempView("customers")

revenue_by_category = spark.sql("""
    SELECT
        hotels.category,
        SUM(bookings.amount) AS total_revenue
    FROM bookings
    INNER JOIN hotels
        ON bookings.hotel_id = hotels.hotel_id
    GROUP BY hotels.category
    ORDER BY total_revenue DESC
""")

revenue_by_category.show()

+--------+--------------------+
|category|       total_revenue|
+--------+--------------------+
|  Luxury|1.3354386324999988E8|
| Premium| 7.115015266000015E7|
|  Budget|2.7711915650000043E7|
+--------+--------------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, dense_rank

# Give aliases to the DataFrames
b = bookings_df.alias("b")
h = hotels_df.alias("h")

# Join and calculate revenue per hotel per city
hotel_revenue = (
    b
    .join(
        h,
        col("b.hotel_id") == col("h.hotel_id"),
        "inner"
    )
    .groupBy(
        col("h.city"),
        col("h.hotel_name")
    )
    .agg(
        sum(col("b.amount")).alias("revenue")
    )
)

# Window function: rank hotels within each city by revenue
window_spec = (
    Window
    .partitionBy("city")
    .orderBy(col("revenue").desc())
)

top_3_hotels_per_city = (
    hotel_revenue
    .withColumn(
        "rank",
        dense_rank().over(window_spec)
    )
    .filter(col("rank") <= 3)
    .select(
        col("city"),
        col("hotel_name"),
        col("revenue"),
        col("rank")
    )
    .orderBy(
        col("city"),
        col("rank")
    )
)

display(top_3_hotels_per_city)

city,hotel_name,revenue,rank
Anantapur,Azure Retreat,1736526.8500000003,1
Anantapur,Lotus Retreat,693900.1900000002,2
Anantapur,Orchid Residency,266747.27999999997,3
Bengaluru,Serene Inn,2558278.320000001,1
Bengaluru,Cedar Resort,2017586.87,2
Bengaluru,Heritage Stay,1751340.0400000005,3
Delhi,Orchid Retreat,3053896.119999999,1
Delhi,Orchid Residency,2874672.199999999,2
Delhi,Marigold Stay,2759378.5399999996,3
Goa,Cedar Residency,5835120.15,1


## Task 7 - Write the result
Write your city-revenue result as **Parquet**, and also as a **Delta table** with
`saveAsTable`. Read the Delta table back to confirm.

In [0]:
# Write city revenue result as Parquet
city_revenue.write.mode("overwrite").parquet(
    f"{BASE}/output/city_revenue_parquet"
)

print(f"Written to {BASE}/output/city_revenue_parquet")


# Write city revenue result as Delta table
city_revenue.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(
    "workspace.default.city_revenue"
)

print("Registered as workspace.default.city_revenue")


# Read Delta table back to confirm
spark.table("workspace.default.city_revenue").show()

Written to /Volumes/workspace/default/staynest/output/city_revenue_parquet
Registered as workspace.default.city_revenue
+---------+------+-------+---------+-------------+----------------+--------------+--------------------+------------------+
|     city|orders|revenue|avg_order|biggest_order|unique_customers|no_of_bookings|       total_revenue|       avg_revenue|
+---------+------+-------+---------+-------------+----------------+--------------+--------------------+------------------+
|      Goa|  NULL|   NULL|     NULL|     78481.24|            1441|          2546| 4.459670178999999E7|17516.379336213664|
|   Mumbai|  NULL|   NULL|     NULL|     78524.52|            1153|          1715|       3.624122112E7| 21131.90735860058|
|    Delhi|  NULL|   NULL|     NULL|     78669.69|             861|          1174| 2.631428154000001E7|22414.209148211252|
|   Jaipur|  NULL|   NULL|     NULL|     76904.87|             796|           979|2.4436853129999984E7| 24961.03486210417|
|Bengaluru|  NULL| 

## Task 8 - One chained pipeline
In a single chain: keep completed bookings, join hotels, keep hotels with
`star_rating >= 4.0`, group by `city`, sum revenue, order descending, take the
top 5. End with one `.show()`.

In [0]:
top5_cities = (
    bookings_df
    .filter(col("status") == "completed")
    .join(
        hotels_df.drop("city"),
        on="hotel_id",
        how="inner"
    )
    .filter(col("star_rating") >= 4.0)
    .groupBy("city")
    .agg(
        sum("amount").alias("total_revenue")
    )
    .orderBy(
        col("total_revenue").desc()
    )
    .limit(5)
)

top5_cities.show()

+---------+--------------------+
|     city|       total_revenue|
+---------+--------------------+
|      Goa|2.4725377899999995E7|
|   Mumbai|1.8937817669999998E7|
|    Delhi|1.8580418800000004E7|
|Bengaluru|    9129192.92999999|
|  Udaipur|          5916270.08|
+---------+--------------------+

